In [1]:
import os
import json
import urllib.request

# 1. Setup local raw data directory relative to the notebooks folder
raw_dir = "../data/raw"
os.makedirs(raw_dir, exist_ok=True)

# Define how many matches you want to download initially
MAX_MATCHES = 50 

print("Fetching competition data to locate available match IDs...")

# Target a major, high-volume league season (La Liga 2017/2018)
competitions_url = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/11/90.json"

try:
    # Read the competition match index file
    with urllib.request.urlopen(competitions_url) as response:
        match_data = json.loads(response.read().decode('utf-8'))
    
    # Extract unique match IDs
    match_ids = [str(match['match_id']) for match in match_data][:MAX_MATCHES]
    print(f"Discovered {len(match_ids)} matches available for downloading.")

    # 2. Iterative Ingestion Loop
    downloaded_count = 0
    for i, match_id in enumerate(match_ids):
        destination_path = os.path.join(raw_dir, f"{match_id}.json")
        
        # Guard clause: avoid re-downloading files that exist locally
        if os.path.exists(destination_path):
            continue
            
        event_url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{match_id}.json"
        
        try:
            urllib.request.urlretrieve(event_url, destination_path)
            downloaded_count += 1
            if downloaded_count % 10 == 0 or i == len(match_ids) - 1:
                print(f"Progress: Checked {i+1}/{len(match_ids)} files... Downloaded {downloaded_count} new matches.")
        except Exception as file_error:
            print(f"Skipping match {match_id} due to an error: {file_error}")
            
    print(f"\n✅ Data collection complete! Local files are saved in: {raw_dir}")

except Exception as global_error:
    print(f"Could not connect to StatsBomb index repository: {global_error}")

Fetching competition data to locate available match IDs...
Discovered 35 matches available for downloading.

✅ Data collection complete! Local files are saved in: ../data/raw


In [2]:
import json
import pandas as pd
import numpy as np

test_match_file = "../data/raw/3764440.json"

#Convert the json file to a pandas dataframe
with open(test_match_file, 'r', encoding='utf-8') as f:
    match_data = json.load(f)

df = pd.DataFrame(match_data)

print(f"Total number of match events logged: {len(df)}")

Total number of match events logged: 4160


In [3]:
df.columns

Index(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type',
       'possession', 'possession_team', 'play_pattern', 'team', 'duration',
       'tactics', 'related_events', 'player', 'position', 'location', 'pass',
       'carry', 'under_pressure', 'dribble', 'shot', 'goalkeeper', 'out',
       'ball_recovery', 'duel', 'ball_receipt', 'clearance', 'off_camera',
       'counterpress', 'interception', 'foul_won', 'foul_committed',
       'substitution', '50_50', 'injury_stoppage'],
      dtype='str')

In [4]:
df.iloc[101]

id                              7971a576-07bb-41c2-8e43-1e51dabfc2b3
index                                                            102
period                                                             1
timestamp                                               00:02:37.544
minute                                                             2
second                                                            37
type                                     {'id': 43, 'name': 'Carry'}
possession                                                         5
possession_team                     {'id': 217, 'name': 'Barcelona'}
play_pattern                       {'id': 1, 'name': 'Regular Play'}
team                                {'id': 217, 'name': 'Barcelona'}
duration                                                    2.215822
tactics                                                          NaN
related_events     [91216b78-846c-46ef-9ccd-8bce0d035861, b459525...
player                 {'id': 5213

In [5]:
df['event_type'] = df['type'].apply(lambda x: x.get('name') if isinstance (x,dict) else None)
df['event_type'].tail()

4155             Pass
4156    Ball Receipt*
4157     Interception
4158         Half End
4159         Half End
Name: event_type, dtype: str

In [6]:
df_pass = df[df['event_type'] == 'Pass'].copy()
df_pass['event_type'].head(10)

4     Pass
7     Pass
10    Pass
13    Pass
16    Pass
21    Pass
24    Pass
27    Pass
30    Pass
33    Pass
Name: event_type, dtype: str

In [7]:
pd.set_option('display.max_colwidth', None)
print(df_pass['pass'].head(50))

4                                                                         {'recipient': {'id': 24517, 'name': 'José Raúl Gutiérrez Parejo'}, 'length': 15.597436, 'angle': 2.428794, 'height': {'id': 1, 'name': 'Ground Pass'}, 'end_location': [49.2, 50.3], 'body_part': {'id': 38, 'name': 'Left Foot'}, 'type': {'id': 65, 'name': 'Kick Off'}}
7                                                                                                                  {'recipient': {'id': 24169, 'name': 'Gonzalo Cacicedo Verdú'}, 'length': 17.520845, 'angle': -2.8342814, 'height': {'id': 1, 'name': 'Ground Pass'}, 'end_location': [33.4, 46.1], 'body_part': {'id': 40, 'name': 'Right Foot'}}
10                                                                                                             {'recipient': {'id': 9857, 'name': 'José Manuel Sánchez Guillén'}, 'length': 27.011848, 'angle': -1.5411754, 'height': {'id': 1, 'name': 'Ground Pass'}, 'end_location': [35.1, 18.3], 'body_part': {'id': 40, 

In [8]:
#I see that when the pass is Complete there is not a outcome written, and when it's incomplete there is
df_pass['outcome'] = df_pass['pass'].apply(lambda x: x.get('outcome') if isinstance (x, dict) else None)
df_pass['outcome'] = df_pass['outcome'].fillna('Complete')
df_pass['outcome'].head()

4     Complete
7     Complete
10    Complete
13    Complete
16    Complete
Name: outcome, dtype: object

In [9]:
df_pass['outcome'] = (df_pass['outcome'] == 'Complete').astype(int)
df_pass['outcome'].head()

4     1
7     1
10    1
13    1
16    1
Name: outcome, dtype: int64

In [10]:
df_pass['outcome'].value_counts(normalize=True)

outcome
1    0.891358
0    0.108642
Name: proportion, dtype: float64

In [11]:
df_pass['under_pressure']

4        NaN
7        NaN
10       NaN
13       NaN
16       NaN
        ... 
4134    True
4137    True
4141    True
4152     NaN
4155     NaN
Name: under_pressure, Length: 1215, dtype: object

In [12]:
#I see that the column is NaN when it's not under pressure and "True" when it is
#I will now make the column binary

df_pass['under_pressure'] = (df_pass['under_pressure']).fillna(False).astype(int)

In [13]:
df_pass['under_pressure']

4       0
7       0
10      0
13      0
16      0
       ..
4134    1
4137    1
4141    1
4152    0
4155    0
Name: under_pressure, Length: 1215, dtype: int64

In [14]:
df_pass['under_pressure'].value_counts(normalize=True)

under_pressure
0    0.898765
1    0.101235
Name: proportion, dtype: float64

In [15]:
df_pass.iloc[0]

id                                                                                                                                                                                                                                                       23743e50-fbe4-4929-938e-9b3ab39ff4ff
index                                                                                                                                                                                                                                                                                       5
period                                                                                                                                                                                                                                                                                      1
timestamp                                                                                                                                     

In [16]:
#I will flatten the data that have a simple dictionary
columns_with_name = ['type', 'possession_team', 'play_pattern', 'team', 'player', 'position']

for col in columns_with_name:
    df_pass[col] = df_pass[col].apply(lambda x: x.get('name') if isinstance (x, dict) else None)

df_pass.iloc[0]

id                                                                                                                                                                                                                                                       23743e50-fbe4-4929-938e-9b3ab39ff4ff
index                                                                                                                                                                                                                                                                                       5
period                                                                                                                                                                                                                                                                                      1
timestamp                                                                                                                                     

In [17]:
df_pass['pass'].iloc[0]

{'recipient': {'id': 24517, 'name': 'José Raúl Gutiérrez Parejo'},
 'length': 15.597436,
 'angle': 2.428794,
 'height': {'id': 1, 'name': 'Ground Pass'},
 'end_location': [49.2, 50.3],
 'body_part': {'id': 38, 'name': 'Left Foot'},
 'type': {'id': 65, 'name': 'Kick Off'}}

In [18]:
#I will now flatten the pass column
#Using the pd.json_normalize() I can do this easier without having to do .apply for everything inisde the column 'pass'
#I tried the above but took way to much time to run, so I will just do the .apply method to every key I need
#I will use functions for my convenience

def apply_simple(df, colname, key):
    return df[colname].apply(lambda x: x.get(key) if isinstance(x,dict) else None)

def apply_complex(df, colname, key1, key2):
    return df[colname].apply(lambda x: x.get(key1, {}).get(key2) if isinstance (x,dict) else None)

In [19]:
df_pass['recipient'] = apply_complex(df_pass, 'pass', 'recipient', 'name')
df_pass['length'] = apply_simple(df_pass, 'pass', 'length')
df_pass['angle'] = apply_simple(df_pass, 'pass', 'angle')
df_pass['height'] = apply_complex(df_pass, 'pass', 'height', 'name')
df_pass['end_location'] = apply_simple(df_pass, 'pass', 'end_location')
df_pass['body_part'] = apply_complex(df_pass, 'pass', 'body_part', 'name')
df_pass['type'] = apply_complex(df_pass, 'pass', 'type', 'name')

In [20]:
df_pass.iloc[0]

id                                                                                                                                                                                                                                                       23743e50-fbe4-4929-938e-9b3ab39ff4ff
index                                                                                                                                                                                                                                                                                       5
period                                                                                                                                                                                                                                                                                      1
timestamp                                                                                                                                     

In [21]:
#I will now get the coordinates for starting and ending location of the pass
df_pass['start_x'] = df_pass['location'].apply(lambda x: x[0] if isinstance(x, list) else None)
df_pass['start_y'] = df_pass['location'].apply(lambda x: x[1] if isinstance(x, list) else None)
df_pass['end_x'] = df_pass['end_location'].apply(lambda x: x[0] if isinstance(x, list) else None)
df_pass['end_y'] = df_pass['end_location'].apply(lambda x: x[1] if isinstance(x, list) else None)

In [22]:
df_pass[['start_x', 'start_y', 'end_x', 'end_y']].head()

,start_x,start_y,end_x,end_y
4,61.0,40.1,49.2,50.3
7,50.1,51.4,33.4,46.1
10,34.3,45.3,35.1,18.3
13,34.5,22.8,26.5,55.5
16,34.1,63.3,89.6,56.1


In [23]:
#Create the function that calculates the angle between two points
def two_point_angle(x1, y1, x2, y2):
    return np.arctan2(y2-y1, x2-x1)

In [24]:
df_pass['angle_calc'] = two_point_angle(df_pass['start_x'], df_pass['start_y'], df_pass['end_x'], df_pass['end_y'])

In [25]:
#Compare between the calculated and given values of the angle
(df_pass['angle_calc'] - df_pass['angle']).describe()

count    1.215000e+03
mean     1.213277e-10
std      5.003024e-08
min     -2.061205e-07
25%     -2.575655e-08
50%      0.000000e+00
75%      2.682618e-08
max      1.940907e-07
dtype: float64

In [26]:
df_pass['angle'] = df_pass['angle_calc']
df_pass.drop(columns=['angle_calc'], inplace=True)

In [27]:
#Create a function for the Pythagorean theorem
def Pythagorean(x1, y1, x2, y2):
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

In [28]:
#Calculate the length of the pass, the distance of the receiver from the goal, and the distance of the receiver from the central line of the pitch
df_pass['length_calc'] = Pythagorean(df_pass['start_x'], df_pass['start_y'], df_pass['end_x'], df_pass['end_y'])
opponentgoal_x = 120
opponentgoal_y = 40
df_pass['dist_receiv_goal'] = Pythagorean(opponentgoal_x, opponentgoal_y, df_pass['end_x'], df_pass['end_y'])
df_pass['dist_receiv_middle'] = df_pass['end_x'] - 60

In [29]:
#Compare the calculated and given values of the length of the pass
(df_pass['length_calc'] - df_pass['length']).describe()

count    1.215000e+03
mean     3.108854e-08
std      6.646040e-07
min     -3.340939e-06
25%     -2.813428e-07
50%      0.000000e+00
75%      3.446706e-07
max      5.793543e-06
dtype: float64

In [30]:
#Drop the given value of the pass
df_pass['length'] = df_pass['length_calc']
df_pass.drop(columns = ['length_calc'], inplace=True)

In [31]:
df_pass.columns

Index(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type',
       'possession', 'possession_team', 'play_pattern', 'team', 'duration',
       'tactics', 'related_events', 'player', 'position', 'location', 'pass',
       'carry', 'under_pressure', 'dribble', 'shot', 'goalkeeper', 'out',
       'ball_recovery', 'duel', 'ball_receipt', 'clearance', 'off_camera',
       'counterpress', 'interception', 'foul_won', 'foul_committed',
       'substitution', '50_50', 'injury_stoppage', 'event_type', 'outcome',
       'recipient', 'length', 'angle', 'height', 'end_location', 'body_part',
       'start_x', 'start_y', 'end_x', 'end_y', 'dist_receiv_goal',
       'dist_receiv_middle'],
      dtype='str')

In [32]:
df_pass[['period', 'timestamp', 'minute', 'second']]

,period,timestamp,minute,second
4,1,00:00:00.438,0,0
7,1,00:00:01.752,0,1
10,1,00:00:04.372,0,4
13,1,00:00:07.325,0,7
16,1,00:00:12.086,0,12
...,...,...,...,...
4134,2,00:47:41.035,92,41
4137,2,00:47:42.211,92,42
4141,2,00:47:44.668,92,44
4152,2,00:47:53.460,92,53


In [33]:
df[['period', 'timestamp', 'minute', 'second', 'event_type']].iloc[10:20]

,period,timestamp,minute,second,event_type
10,1,00:00:04.372,0,4,Pass
11,1,00:00:06.078,0,6,Ball Receipt*
12,1,00:00:06.078,0,6,Carry
13,1,00:00:07.325,0,7,Pass
14,1,00:00:09.623,0,9,Ball Receipt*
15,1,00:00:09.623,0,9,Carry
16,1,00:00:12.086,0,12,Pass
17,1,00:00:15.148,0,15,Ball Receipt*
18,1,00:00:15.148,0,15,Miscontrol
19,1,00:00:18.297,0,18,Ball Recovery


In [34]:
df['related_events'].tail(10)

4150                                                                             NaN
4151    [1feb6155-826f-478d-9c54-e0c952bb4db6, 3b796407-f7e5-4005-91bd-bffdc062e9d6]
4152                                          [a62eb4b2-636b-43e3-8d98-55b1c5468a34]
4153                                          [3b796407-f7e5-4005-91bd-bffdc062e9d6]
4154    [2c8927c9-cc55-4ced-b799-caf207bcc23e, a62eb4b2-636b-43e3-8d98-55b1c5468a34]
4155    [55878551-e34f-4ad0-9910-271004ff7813, b64041a8-55aa-40fa-865b-a71ee37ddedd]
4156                                          [2c8927c9-cc55-4ced-b799-caf207bcc23e]
4157                                          [2c8927c9-cc55-4ced-b799-caf207bcc23e]
4158                                          [390f9019-f867-4a61-ad12-b780510f81d3]
4159                                          [b1510d0a-19e9-4b77-b208-62d89a116288]
Name: related_events, dtype: object

In [35]:
print(df.groupby('period')[['minute', 'second']].agg(['min', 'max']))

       minute     second    
          min max    min max
period                      
1           0  45      0  59
2          45  92      0  59


In [36]:
df[['second', 'minute']].iloc[50:60]

,second,minute
50,4,1
51,4,1
52,5,1
53,5,1
54,6,1
55,6,1
56,6,1
57,7,1
58,7,1
59,7,1


In [37]:
#Calculate the time of each event in seconds
df['raw_seconds'] = 60 * df['minute'] + df['second']
df['raw_seconds'].iloc[50:60]

50    64
51    64
52    65
53    65
54    66
55    66
56    66
57    67
58    67
59    67
Name: raw_seconds, dtype: int64

In [38]:
half_seconds = 45*60
df['half_percentage'] = np.minimum(df['raw_seconds']/half_seconds, 1.0)
df['half_percentage'].iloc[50:60]

50    0.023704
51    0.023704
52    0.024074
53    0.024074
54    0.024444
55    0.024444
56    0.024444
57    0.024815
58    0.024815
59    0.024815
Name: half_percentage, dtype: float64

In [39]:
injury_zone = df[(df['minute'] >= 45) & (df['minute'] <= 50)]
print(injury_zone[['period', 'minute', 'second', 'raw_seconds', 'half_percentage']].head(10))

      period  minute  second  raw_seconds  half_percentage
1977       1      45       2         2702              1.0
1978       1      45       3         2703              1.0
1979       1      45       3         2703              1.0
1980       1      45       4         2704              1.0
1981       1      45       5         2705              1.0
1982       1      45       5         2705              1.0
1983       1      45      32         2732              1.0
1984       1      45      33         2733              1.0
1985       1      45      33         2733              1.0
1986       1      45      34         2734              1.0


In [40]:
df['team'].iloc[50:60]

50    {'id': 217, 'name': 'Barcelona'}
51    {'id': 217, 'name': 'Barcelona'}
52       {'id': 1042, 'name': 'Elche'}
53    {'id': 217, 'name': 'Barcelona'}
54    {'id': 217, 'name': 'Barcelona'}
55    {'id': 217, 'name': 'Barcelona'}
56    {'id': 217, 'name': 'Barcelona'}
57       {'id': 1042, 'name': 'Elche'}
58    {'id': 217, 'name': 'Barcelona'}
59    {'id': 217, 'name': 'Barcelona'}
Name: team, dtype: object

In [41]:
df['team'] = df['team'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)
df['team'].iloc[50:60]

50    Barcelona
51    Barcelona
52        Elche
53    Barcelona
54    Barcelona
55    Barcelona
56    Barcelona
57        Elche
58    Barcelona
59    Barcelona
Name: team, dtype: str

In [42]:
unique_teams = df['team'].dropna().unique()
team1 = unique_teams[0]
team2 = unique_teams[1]
print(team1, team2)

Barcelona Elche


In [43]:
print(df['event_type'].unique())

<StringArray>
[      'Starting XI',        'Half Start',              'Pass',
     'Ball Receipt*',             'Carry',        'Miscontrol',
     'Ball Recovery',          'Pressure',     'Dribbled Past',
           'Dribble',              'Shot',       'Goal Keeper',
              'Duel',         'Clearance',      'Dispossessed',
      'Interception',    'Foul Committed',          'Foul Won',
             'Block',   'Injury Stoppage', 'Referee Ball-Drop',
        'Player Off',         'Player On',          'Half End',
      'Substitution',    'Tactical Shift',             '50/50']
Length: 27, dtype: str


In [44]:
print(df[df['event_type']=='Shot']['shot'].iloc[0])

{'statsbomb_xg': 0.06967228, 'end_location': [117.3, 43.2, 0.1], 'key_pass_id': '44624288-c019-4b2d-980a-1b4d62bd7c8f', 'technique': {'id': 93, 'name': 'Normal'}, 'body_part': {'id': 38, 'name': 'Left Foot'}, 'type': {'id': 87, 'name': 'Open Play'}, 'outcome': {'id': 100, 'name': 'Saved'}, 'freeze_frame': [{'location': [111.3, 43.5], 'player': {'id': 4447, 'name': 'Martin Braithwaite Christensen'}, 'position': {'id': 21, 'name': 'Left Wing'}, 'teammate': True}, {'location': [97.4, 54.5], 'player': {'id': 8118, 'name': 'Frenkie de Jong'}, 'position': {'id': 13, 'name': 'Right Center Midfield'}, 'teammate': True}, {'location': [100.2, 40.6], 'player': {'id': 30486, 'name': 'Pedro González López'}, 'position': {'id': 15, 'name': 'Left Center Midfield'}, 'teammate': True}, {'location': [87.3, 64.0], 'player': {'id': 43728, 'name': 'Óscar Mingueza García'}, 'position': {'id': 2, 'name': 'Right Back'}, 'teammate': True}, {'location': [110.2, 57.7], 'player': {'id': 5503, 'name': 'Lionel Andr

In [45]:
df['shot_outcome'] = df['shot'].apply(lambda x: x.get('outcome', {}).get('name') if isinstance(x,dict) else None)
df['shot_outcome'].dropna()

63        Saved
180       Off T
938       Saved
1479      Off T
1873      Saved
2146       Goal
2229      Off T
2291    Blocked
2928    Blocked
3113       Goal
3270       Goal
3365      Off T
3665      Off T
4033    Wayward
4128    Blocked
Name: shot_outcome, dtype: str

In [46]:
#Find the moments when a goal is scored for each team
df['team1_goal_step'] = np.where((df['team'] == team1) & (df['shot_outcome'] == 'Goal'), 1, 0)
df['team2_goal_step'] = np.where((df['team'] == team2) & (df['shot_outcome'] == 'Goal'), 1, 0)
df['team1_goal_step'].iloc[2140:2150]

2140    0
2141    0
2142    0
2143    0
2144    0
2145    0
2146    1
2147    0
2148    0
2149    0
Name: team1_goal_step, dtype: int64

In [47]:
#Creating a column with the running goals scored for each time
df['team1_running_score'] = df['team1_goal_step'].cumsum()
df['team2_running_score'] = df['team2_goal_step'].cumsum()
df['team1_running_score'].iloc[2140:2150]

2140    0
2141    0
2142    0
2143    0
2144    0
2145    0
2146    1
2147    1
2148    1
2149    1
Name: team1_running_score, dtype: int64

In [61]:
#Create the net_score for each event, taking into account the team that makes the current event
"""
for index, row in df.iterrows():
    if row['team'] == team1:
        row['net_score'] = df['team1_running_score'] - df['team2_running_score']
    elif row['team'] == team2:
        row['net_score'] = df['team2_running_score'] - df['team1_running_score']
"""
#The above is too slow
#I will use a vectorized approach

conditions = [
    df['team'] == team1,
    df['team'] == team2
]

choices = [
    df['team1_running_score'] - df['team2_running_score'],
    df['team2_running_score'] - df['team1_running_score']
]

df['net_score'] = np.select(conditions, choices, default=0)

df['net_score'].tail()

4155    3
4156    3
4157   -3
4158    3
4159   -3
Name: net_score, dtype: int64

In [76]:
#Pass the latest columns generated to the df_pass
df_pass['half_percentage'] = df['half_percentage']
df_pass['net_score'] = df['net_score']

In [77]:
print(df_pass[['team', 'period', 'minute', 'half_percentage', 'net_score']].iloc[630:640])

           team  period  minute  half_percentage  net_score
2130  Barcelona       2      46              1.0          0
2133  Barcelona       2      46              1.0          0
2136  Barcelona       2      46              1.0          0
2140  Barcelona       2      47              1.0          0
2142  Barcelona       2      47              1.0          0
2148      Elche       2      48              1.0         -1
2150      Elche       2      48              1.0         -1
2152  Barcelona       2      48              1.0          1
2156  Barcelona       2      48              1.0          1
2159      Elche       2      48              1.0         -1
